# LLM Call Isolation — 3R-assist
Test each LLM call in the retrieval pipeline independently.
Make sure `ollama serve` is running before executing any cell.

In [ ]:
import sys, json
sys.path.insert(0, '..')  # add backend/ to path

from app.adapters.llm import OllamaLLMAdapter

MODEL = 'qwen2.5:14b'
llm = OllamaLLMAdapter(model=MODEL)

print(f'Adapter ready: {MODEL}')

---
## Shared protocol text
Edit this cell to change the input for all tests below.

In [ ]:
Project text = """
# 
"""

print(PROTOCOL_TEXT.strip())

---
## 0 — Parameter Extraction
The LLM reads the protocol and extracts structured parameters.
These are used by all subsequent steps.

In [ ]:
from app.prompts.extraction import build_extraction_prompt

prompt = build_extraction_prompt(PROTOCOL_TEXT)
print('=== PROMPT ===\n')
print(prompt)

In [ ]:
raw = llm.call(prompt, max_tokens=2048, json_mode=True)
print('=== RAW RESPONSE ===\n')
print(raw)

In [ ]:
from app.adapters.llm import _parse_json_payload, _raw_experiments_from_payload
from app.services.extraction import enrich_raw_experiments, build_analyze_response

payload = _parse_json_payload(raw)
raw_experiments = _raw_experiments_from_payload(payload, raw_response=raw)

print('=== EXTRACTED (raw) ===\n')
print(json.dumps(payload, indent=2))

In [ ]:
from app.adapters.llm import ExtractionError

if isinstance(raw_experiments, ExtractionError):
    print('EXTRACTION FAILED:', raw_experiments.message)
else:
    result = build_analyze_response(enrich_raw_experiments(raw_experiments))
    PARAMS = result.params

    print('=== EXTRACTED PARAMETERS ===\n')
    print(f'  endpoint_category : {PARAMS.endpoint_category}')
    print(f'  study_domain      : {PARAMS.study_domain}')
    print(f'  species           : {PARAMS.species}')
    print(f'  route             : {PARAMS.route}')
    print(f'  procedure_text    : {PARAMS.procedure_text}')
    print(f'  regulatory        : {PARAMS.regulatory}')

---
## 1 — Necessity Assessment

In [ ]:
from pubmed.prompts.necessity import build_necessity_prompt

prompt = build_necessity_prompt(
    protocol_text=PROTOCOL_TEXT,
    endpoint_category=PARAMS.endpoint_category,
    study_domain=PARAMS.study_domain,
    species=PARAMS.species,
    route=PARAMS.route,
    procedure_text=PARAMS.procedure_text,
    regulatory=PARAMS.regulatory,
)

print('=== PROMPT ===\n')
print(prompt)

In [ ]:
raw = llm.call(prompt, max_tokens=1024, json_mode=True)
print('=== RAW RESPONSE ===\n')
print(raw)

In [ ]:
necessity = json.loads(raw)
print(json.dumps(necessity, indent=2))

---
## 2 — Search Plan Generation (Path A + B queries)

In [ ]:
from pubmed.prompts.alternative_query import build_alternative_query_prompt

prompt = build_alternative_query_prompt(
    protocol_text=PROTOCOL_TEXT,
    endpoint_category=PARAMS.endpoint_category,
    study_domain=PARAMS.study_domain,
    species=PARAMS.species,
    route=PARAMS.route,
    procedure_text=PARAMS.procedure_text,
)

print('=== PROMPT ===\n')
print(prompt)

In [ ]:
raw = llm.call(prompt, max_tokens=512, json_mode=True)
print('=== RAW RESPONSE ===\n')
print(raw)

In [ ]:
plan = json.loads(raw)
print(json.dumps(plan, indent=2))

---
## 3 — Ranking
Fetch real candidates from the database, then rank them.

In [ ]:
import asyncio
from app.adapters.embedder import SentenceTransformerEmbedder
from pubmed.db.repository import PubMedRepository

embedder = SentenceTransformerEmbedder('all-MiniLM-L6-v2')
repo = PubMedRepository()

SEARCH_QUERY = plan.get('endpoint_search_query', 'skin irritation alternative in vitro assay')
print('Search query:', SEARCH_QUERY)

embedding = embedder.embed(SEARCH_QUERY)
rows = await repo.search_by_endpoint_embedding(embedding, top_k=5)

CANDIDATES = [
    {
        'pmid': rec.pmid,
        'title': rec.title,
        'abstract_text': (rec.abstract_text or '')[:400],
        'source_class': None,
    }
    for rec, score in rows
]

print(f'\nFetched {len(CANDIDATES)} candidates:')
for c in CANDIDATES:
    print(f"  PMID {c['pmid']}: {c['title'][:80]}")

In [ ]:
from pubmed.prompts.ranking import build_ranking_prompt

prompt = build_ranking_prompt(
    endpoint_category=PARAMS.endpoint_category,
    study_domain=PARAMS.study_domain,
    procedure_text=PARAMS.procedure_text,
    candidates=CANDIDATES,
)

print('=== PROMPT ===\n')
print(prompt)

In [ ]:
raw = llm.call(prompt, max_tokens=768, json_mode=True)
print('=== RAW RESPONSE ===\n')
print(raw)

In [ ]:
ranking = json.loads(raw)
print(json.dumps(ranking, indent=2))

---
## 4 — Summary Generation

In [ ]:
from pubmed.prompts.summary import build_summary_prompt

rec_by_pmid = {c['pmid']: c for c in CANDIDATES}
ranked_items = [
    {
        'pmid': r['pmid'],
        'title': rec_by_pmid[r['pmid']]['title'] if r['pmid'] in rec_by_pmid else '',
        'abstract_text': rec_by_pmid[r['pmid']]['abstract_text'] if r['pmid'] in rec_by_pmid else '',
        'three_r_class': r.get('three_r_class', 'refinement'),
        'relevance_explanation': r.get('relevance_explanation', ''),
        'rank': i + 1,
    }
    for i, r in enumerate(ranking.get('ranked', []))
    if r.get('include') and r.get('pmid') in rec_by_pmid
]

print(f'{len(ranked_items)} included recommendations')

prompt = build_summary_prompt(
    endpoint_category=PARAMS.endpoint_category,
    study_domain=PARAMS.study_domain,
    procedure_text=PARAMS.procedure_text,
    recommendations=ranked_items,
)

print('\n=== PROMPT ===\n')
print(prompt)

In [ ]:
raw = llm.call(prompt, max_tokens=400, json_mode=True)
print('=== RAW RESPONSE ===\n')
print(raw)

In [ ]:
summary = json.loads(raw)
print('SUMMARY:\n')
print(summary.get('summary', ''))
print('\nCITED PMIDs:', summary.get('cited_pmids', []))